In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"


import torch
# torch.cuda.init()
print(os.environ.get("CUDA_VISIBLE_DEVICES"))
!source /home/jupyter/Mrigi/env.sh
hf_token = os.environ.get("HF_TOKEN")

torch.cuda.set_device(0)
import json

from langchain.embeddings import HuggingFaceEmbeddings
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.vectorstores import FAISS
from langchain.prompts import ChatPromptTemplate
from langchain.llms import HuggingFacePipeline
from langchain.chains import LLMChain
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline



3


In [3]:
from pydantic import BaseModel, Field
from typing import Optional, Union


In [4]:
!nvidia-smi

Mon Aug  4 20:51:57 2025       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 495.29.05    Driver Version: 495.29.05    CUDA Version: 11.5     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA RTX A5000    Off  | 00000000:31:00.0 Off |                  Off |
| 30%   32C    P8    22W / 230W |  17689MiB / 24256MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
|   1  NVIDIA RTX A5000    Off  | 00000000:4B:00.0 Off |                  Off |
| 30%   

In [5]:
!kill -9 1472362


/bin/bash: line 0: kill: (1472362) - No such process


In [6]:
# ---------- Step 1: Load and Process the JSON Records ----------
# Load JSON records from file. Right now it only has 100 or so papers
with open('filtered_records_on_zeolite.json', 'r') as file:
    filtered_records = json.load(file)

In [7]:
# Process each record: extract paragraphs with "Experimental" supersection,
# concatenate them, and append DOI info.
documents = []
metadata = []

In [8]:
# for record in filtered_records:
#     experimental_paragraphs = [
#         para.get("text", "")
#         for para in record.get("paragraphs", [])
#         if para.get("supersection_name", "").strip() == "Experimental"
#     ]
    
#     # Only add papers that have experimental sections
#     if experimental_paragraphs:
#         combined_text = "\n\n".join(experimental_paragraphs)
#         doi = record.get("doi", "Unknown DOI")
#         combined_text += f"\n\nThis information is from DOI: {doi}"
#         documents.append(combined_text)
#         metadata.append({"doi": doi})

# print(f"Total papers with experimental sections: {len(documents)}")



# for record in filtered_records:
#     abstract_text = record.get("abstract", "").strip()

#     # Only add records that have an abstract
#     if abstract_text:
#         doi = record.get("doi", "Unknown DOI")
#         combined_text = f"{abstract_text}\n\nThis information is from DOI: {doi}"
#         documents.append(combined_text)
#         metadata.append({"doi": doi})

# print(f"Total papers with abstracts: {len(documents)}")



for record in filtered_records:
    doi = record.get("doi", "Unknown DOI")

    # Add abstract as a separate document
    abstract_text = record.get("abstract", "").strip()
    if abstract_text:
        combined_text = f"{abstract_text}\n\nThis information is from DOI: {doi}"
        documents.append(combined_text)
        metadata.append({"doi": doi, "source": "abstract"})

    # Add each paragraph as a separate document
    for para in record.get("paragraphs", []):
        paragraph_text = para.get("text", "").strip()
        if paragraph_text:
            combined_text = f"{paragraph_text}\n\nThis information is from DOI: {doi}"
            documents.append(combined_text)
            metadata.append({"doi": doi, "source": "paragraph"})

print(f"Total individual documents (abstracts + paragraphs): {len(documents)}")


Total individual documents (abstracts + paragraphs): 1474439


In [9]:
# for doc in documents:
#     print(doc)
#     print("-"*40)  # Optional: adds a visual separator between different papers


In [10]:
# ---------- Step 2: Create the Vector Database with FAISS and SciBERT ----------
# Use SciBERT for embeddings via HuggingFaceEmbeddings.
# embeddings = HuggingFaceEmbeddings(model_name="allenai/scibert_scivocab_uncased")
embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

In [ ]:
# Build a FAISS vector store where each document represents a paper’s experimental section.
# vector_db = FAISS.from_texts(documents, embeddings, metadatas=metadata)
# vector_db = FAISS.from_texts(documents, embedding=embeddings, metadata=metadata)

from langchain.vectorstores import FAISS

batch_size = 50
vector_db = None

for i in range(0, len(documents), batch_size):
    docs_batch = documents[i:i + batch_size]
    meta_batch = metadata[i:i + batch_size]

    if vector_db is None:
        vector_db = FAISS.from_texts(docs_batch, embedding=embeddings, metadatas=meta_batch)
    else:
        vector_db.add_texts(docs_batch, metadata=meta_batch)


#try Chroma

In [ ]:
# ---------- Step 3: Use the Vector DB in a RAG Pipeline ----------
# Define your query.
# query = "How is hierarchical ZSM-5 synthesized?"
# query = "What is the best way to synthesize ZSM-5 that is hierarchical?"
# query = "How is Silicalite-1 synthesized?"
query = "How is ZSM-5 synthesized?"


In [ ]:
# Retrieve top k relevant documents (papers) from the vector database.
retrieved_docs = vector_db.similarity_search(query, k=1)
context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])

In [ ]:
# print(embeddings.embed_query("Silicalite-1 synthesis"))
# print(embeddings.embed_documents(["test text 1", "test text 2"]))


In [ ]:
context_text

In [ ]:
# Define a prompt template that instructs the LLM how to answer.
PROMPT_TEMPLATE = """
Answer the question based only on the following context:
{context}
Answer the question based on the above context: {question}.
Provide a detailed answer.
Provide which DOI the answer is retrieved from.
"""

In [ ]:
prompt_template = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)
prompt = prompt_template.format(context=context_text, question=query)

In [ ]:
# ---------- Step 4: Generate an Answer Using an Open-Source LLM ----------
# Setup the open source LLM using Mistral (ensure you have the model or access to it)
model_name = "mistralai/Mistral-7B-Instruct-v0.1"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, use_fast=False, use_auth_token=hf_token)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", trust_remote_code=True, use_auth_token=hf_token, torch_dtype=torch.float16)

In [ ]:
hf_pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer, max_length=10000)

In [ ]:
# Wrap the Hugging Face pipeline with LangChain's HuggingFacePipeline interface.
llm = HuggingFacePipeline(pipeline=hf_pipeline)


In [ ]:
# Generate the answer based on the prompt that includes retrieved context.
response_text = llm(prompt)
print("Response:")
print(response_text)

In [ ]:
class ZeosynRecord(BaseModel):
    doi: str  # always present

    # --- Composition (float) ---
    Si: Optional[float] = None
    Al: Optional[float] = None
    P: Optional[float] = None
    Na: Optional[float] = None
    K: Optional[float] = None
    Li: Optional[float] = None
    Sr: Optional[float] = None
    Rb: Optional[float] = None
    Cs: Optional[float] = None
    Ba: Optional[float] = None
    Ca: Optional[float] = None
    F: Optional[float] = None
    Ge: Optional[float] = None
    Ti: Optional[float] = None
    In: Optional[float] = None
    B: Optional[float] = None
    Mg: Optional[float] = None
    Ga: Optional[float] = None
    Ni: Optional[float] = None
    Mn: Optional[float] = None
    Fe: Optional[float] = None
    Co: Optional[float] = None
    Cr: Optional[float] = None
    Zn: Optional[float] = None
    Nb: Optional[float] = None
    Be: Optional[float] = None
    W: Optional[float] = None
    Ce: Optional[float] = None
    Cu: Optional[float] = None
    Sn: Optional[float] = None
    Gd: Optional[float] = None
    La: Optional[float] = None
    Y: Optional[float] = None
    Dy: Optional[float] = None
    Sm: Optional[float] = None
    Ag: Optional[float] = None
    Cd: Optional[float] = None
    Zr: Optional[float] = None
    V: Optional[float] = None

    # --- Synthesis parameters ---
    H2O: Optional[str] = None
    solvent: Optional[str] = None
    sda1: Optional[str] = None
    sda2: Optional[str] = None
    sda3: Optional[str] = None
    acid: Optional[float] = None
    Ta: Optional[float] = None
    Ru: Optional[float] = None
    Hf: Optional[float] = None
    Yb: Optional[float] = None
    Tl: Optional[float] = None
    As: Optional[float] = None
    normed: Optional[float] = None
    seed: Optional[float] = None
    OH: Optional[float] = None
    aging_time: Optional[float] = None
    aging_temp: Optional[float] = None
    cryst_time: Optional[float] = None
    cryst_temp: Optional[float] = None
    rotation: Optional[float] = None
    Seed_type: Optional[str] = None
    react_vol: Optional[float] = None
    pH: Optional[str] = None
    osda1: Optional[str] = None
    osda2: Optional[str] = None
    osda3: Optional[str] = None
    product1: Optional[str] = None
    product2: Optional[str] = None
    product3: Optional[str] = None
    precursors: Optional[str] = None
    brands: Optional[str] = None
    Si_Al: Optional[float] = Field(None, alias="Si/Al")
    yield_: Optional[float] = Field(None, alias="yield")
    percent_cryst: Optional[float] = Field(None, alias="percent cryst")
    crystal_size: Optional[float] = Field(None, alias="crystal size")
    micropore_volume: Optional[float] = Field(None, alias="micropore volume")
    micropore_diameter: Optional[float] = Field(None, alias="micropore diameter")
    bet_area: Optional[float] = Field(None, alias="bet area")
    external_surface_area: Optional[float] = Field(None, alias="external surface area")
    Notes: Optional[str] = None
    title: Optional[str] = None
    abstract_keywords: Optional[str] = None
    recipe_keywords: Optional[str] = None
    osda1_synonyms: Optional[str] = Field(None, alias="osda1 synonyms")
    osda2_synonyms: Optional[str] = Field(None, alias="osda2 synonyms")
    osda3_synonyms: Optional[str] = Field(None, alias="osda3 synonyms")
    osda1_iupac: Optional[str] = Field(None, alias="osda1 iupac")
    osda2_iupac: Optional[str] = Field(None, alias="osda2 iupac")
    osda3_iupac: Optional[str] = Field(None, alias="osda3 iupac")
    osda1_smiles: Optional[str] = Field(None, alias="osda1 smiles")
    osda2_smiles: Optional[str] = Field(None, alias="osda2 smiles")
    osda3_smiles: Optional[str] = Field(None, alias="osda3 smiles")
    osda1_formula: Optional[str] = Field(None, alias="osda1 formula")
    osda2_formula: Optional[str] = Field(None, alias="osda2 formula")
    osda3_formula: Optional[str] = Field(None, alias="osda3 formula")
    Code1: Optional[str] = None
    Code2: Optional[str] = None
    Code3: Optional[str] = None
    year: Optional[int] = None

In [ ]:
# Define a prompt template that instructs the LLM how to answer.
schema_json = ZeosynRecord.schema_json(indent=2)

PROMPT_TEMPLATE_FMT = """
You are a data-entry assistant.

Given:
---
{raw_answer}
---

Return a JSON object that matches this schema example:

Given: TNU-9 is synthesized using 1,4-bis-(N-methylpyrrolidinium)butane and Na+ cations as the structure-directing agents according to a procedure described elsewhere.

Schema: "product": "TNU-9", "OSDA": "1,4-bis-(N-methylpyrrolidinium)butane", "SDA": Na+ cations



Use `null` for any unknown field.  
Do not add keys that are not in the schema.  
Strings should be exactly as written in the given answer, no extra punctuation.
- Answer only with the JSON and no additional disclaimers, signatures, or footers.
"""

In [ ]:
# response_text_json = llm(
#     prompt_template.format(
#         raw_answer=response_text,
#         ZeosynRecord=ZeosynRecord
#     )
# )
# print("Response:")
# print(response_text_json)
fmt_prompt  = ChatPromptTemplate.from_template(PROMPT_TEMPLATE_FMT)
format_chain = LLMChain(llm=llm, prompt=fmt_prompt)

# 4) Now this will work:
response_text_json = format_chain.run(
    raw_answer   = response_text
)
print(response_text_json)